In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

In [2]:
import tensorflow as tf
import numpy as np


2023-08-13 18:03:45.188777: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-08-13 18:03:45.887986: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [3]:
input_file = 'input.wav'
output_file = 'output.wav'

input_wav = tf.io.read_file(input_file)
output_wav = tf.io.read_file(output_file)

2023-08-13 18:03:46.813823: E tensorflow/compiler/xla/stream_executor/cuda/cuda_driver.cc:266] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


In [4]:
channels = 1

input_lookback = 441

In [5]:
input_samples, sample_rate = tf.audio.decode_wav(input_wav)
output_samples, sample_rate = tf.audio.decode_wav(output_wav)
sample_len = 44100 * 10 #min(len(input_samples), len(output_samples)) 

In [6]:
input_data = np.zeros((sample_len, input_lookback, 1))
output_data = np.zeros((sample_len, 1))

skip = 44100 * 30

for i in range(sample_len):
    input_data[i] = input_samples[skip + i : skip + i + input_lookback]
    output_data[i] = output_samples[skip + i + input_lookback - 1]

np.save('input.npy', input_data)
np.save('output.npy', output_data)

# input_data = np.load('input.npy')
# output_data = np.load('output.npy') 

In [12]:
model = tf.keras.Sequential([
  tf.keras.layers.Flatten(),
  tf.keras.layers.Dense(16, activation='relu'),
  tf.keras.layers.Dense(1)
])

In [13]:
model.compile(optimizer='rmsprop', loss='mse', metrics=['mae'])

In [14]:
history = model.fit(
    x = input_data,
    y = output_data,
    validation_split = 0.1,
    shuffle = True,
    epochs=3)



Epoch 1/3
12404/12404 [==============================] - 23s 2ms/step - loss: 2.0872e-04 - mae: 0.0100 - val_loss: 3.4971e-04 - val_mae: 0.0130
Epoch 2/3
12404/12404 [==============================] - 22s 2ms/step - loss: 1.4627e-04 - mae: 0.0082 - val_loss: 3.4020e-04 - val_mae: 0.0134
Epoch 3/3
12404/12404 [==============================] - 22s 2ms/step - loss: 1.2813e-04 - mae: 0.0077 - val_loss: 2.2327e-04 - val_mae: 0.0105


In [15]:
model.summary()

prediction = model.predict(input_data)

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten_1 (Flatten)         (None, 441)               0         
                                                                 
 dense_2 (Dense)             (None, 16)                7072      
                                                                 
 dense_3 (Dense)             (None, 1)                 17        
                                                                 
Total params: 7,089
Trainable params: 7,089
Non-trainable params: 0
_________________________________________________________________
13782/13782 [==============================] - 17s 1ms/step


In [11]:
prediction_audio = tf.audio.encode_wav(prediction, 44100)
tf.io.write_file('prediction.wav', prediction_audio)